In [14]:
#For /pathto use your own drive path.
#The following cell will connect your google drive to collab
#From there you have to put your specific paths to each file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import shutil

# 1. Copy RAR from Drive to Colab's local VM
print("Copying RAR file...")
!cp "/pathto/Images.rar" "/content/Images.rar"

# 2. Install unrar, create destination directory, and then unrar it silently (x extracts files with full paths)
print("Installing unrar and unzipping images...")
!apt-get install unrar -qq
!mkdir -p "/content/scut_images"
!unrar x -o+ "/content/Images.rar" "/content/scut_images"

print("✅ Done! Images are ready at /content/scut_images")

In [ ]:
# Install libraries first
!pip install deepface pandas scikit-learn

In [ ]:
import os
import pandas as pd
import numpy as np
from deepface import DeepFace
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

CSV_FILE = "pathto/clean_caucasian_averages.csv"

# We read images from the LOCAL Colab folder we just unzipped
# Note: The unzip command often creates a nested folder.
# Check the side panel to see if it's "/content/scut_images" or "/content/scut_images/Images"
IMAGES_FOLDER = "/content/scut_images/Images"
MODEL_FILENAME = "pathto/save/rating_model.pkl" # Save

def train():
    print("Loading data...")
    df = pd.read_csv(CSV_FILE)

    X = []
    y = []

    successful = 0

    for index, row in df.iterrows():
        filename = row['filename']
        avg_score = row['average_score']

        full_path = os.path.join(IMAGES_FOLDER, filename)

        if not os.path.exists(full_path):
            full_path = os.path.join(IMAGES_FOLDER, "Images", filename)
            if not os.path.exists(full_path):
                continue

        try:
            embedding_objs = DeepFace.represent(
                img_path=full_path,
                model_name="Facenet",
                enforce_detection=False
            )
            embedding = embedding_objs[0]["embedding"]

            X.append(embedding)
            y.append(float(avg_score) * 2.0) # 1-10 Scale

            successful += 1
            if successful % 100 == 0:
                print(f"Processed {successful} images...")

        except:
            pass

    print(f"Training on {len(X)} images...")

    # Train
    X = np.array(X)
    y = np.array(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model = LinearRegression()
    model.fit(X_train, y_train)

    error = mean_absolute_error(y_test, model.predict(X_test))
    print(f"Mean Absolute Error: {error:.2f}")

    # Save Final Model back to Google Drive
    final_model = LinearRegression()
    final_model.fit(X, y)
    joblib.dump(final_model, MODEL_FILENAME)
    print(f"💾 Model saved to Google Drive: {MODEL_FILENAME}")

if __name__ == "__main__":
    train()

In [13]:
import joblib
import numpy as np
from deepface import DeepFace

# 1. Load the trained model
print("Loading model...")
model = joblib.load("/pathto/rating_model.pkl")

# 2. Point to a NEW image
img_path = "pathto/model.jpg"

try:
    print(f"Analyzing {img_path}...")

    # 3. Get the Embedding (The input)
    # Note: We must use the exact same settings as training
    embedding_objs = DeepFace.represent(
        img_path=img_path,
        model_name="Facenet",
        enforce_detection=True
    )
    embedding = embedding_objs[0]["embedding"]

    # 4. Predict the Score (The output)
    # Reshape because the model expects a list of inputs, not just one
    prediction = model.predict([embedding])

    score = prediction[0]

    # 5. Clamp the result (Safety check)
    # Sometimes math does weird things like 10.2 or -0.5
    final_score = max(1.0, min(10.0, score))

    print(f"-------- RESULT --------")
    print(f"Predicted Score: {final_score:.2f} / 10.0")
    print(f"------------------------")

except Exception as e:
    print(f"Error: {e}")

Loading model...
Analyzing /content/drive/MyDrive/FaceApp_Project/model.jpg...
-------- RESULT --------
Predicted Score: 8.24 / 10.0
------------------------
